# SOXX 2018–2022 披露集中度 Qlib 训练来源衔接

`plan.json` 固定 2018 年前置日历和 2019–2022 原 Stage1 训练日历，新增唯一 2018-09 原始披露事件，严格在 SEC 申报日期后的实际交易日生效。完整保存 Qlib DatasetH 和 Recorder。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2018_2022_concentration_bridge_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
s=plan['sources'];assert sha(root/s['prior_bridge_summary'])==s['prior_bridge_summary_sha256'] and sha(root/s['2018_Sep_numeric_result'])==s['2018_Sep_numeric_result_sha256']
prior=json.loads((root/s['prior_bridge_summary']).read_text());new=json.loads((root/s['2018_Sep_numeric_result']).read_text())
assert prior['recorder_id']==s['prior_bridge_recorder_id'] and prior['nas_evidence']['sha256']==s['prior_bridge_recorder_NAS_sha256'] and prior['nas_readback']['verified']
assert new['status']=='2018_Sep_original_SOXX_reported_equity_values_closed' and new['parsed_equity_rows']==30 and new['total_closure_difference_usd']==0
state={'status':'started','new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['availability_calendar_span'][0],end_time=plan['availability_calendar_span'][1],freq='day'))
with R.uri_context(prior['recorder_uri']):
 source_rec=R.get_recorder(recorder_id=prior['recorder_id'],experiment_id=prior['experiment_id']);previous=source_rec.load_object('analysis/source_events')
assert len(previous)==15
one=pd.DataFrame([{'accession':new['accession'],'report_period':new['report_period'],'filed':new['filed'],'form':'legacy_old_form','top5':new['top5_disclosed_equity_value_share'],'HHI':new['herfindahl_sum_squared_disclosed_equity_weights'],'source_sha256':new['raw_sha256']}])
events=pd.concat([one,previous[['accession','report_period','filed','form','top5','HHI','source_sha256']]],ignore_index=True).sort_values('filed').reset_index(drop=True)
assert len(events)==16 and events.accession.is_unique and events.report_period.is_unique
events['filed_date']=pd.to_datetime(events.filed)
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
signals=calendar[(calendar>=pd.Timestamp(plan['late_train_span'][0]))&(calendar<=pd.Timestamp(plan['late_train_span'][1]))]
mapped=pd.merge_asof(pd.DataFrame({'signal_date':signals}),events[['effective_signal_date','accession','report_period','filed_date','form','top5','HHI']],left_on='signal_date',right_on='effective_signal_date',direction='backward')
mapped['known']=mapped.accession.notna();mapped['source_age_days']=(mapped.signal_date-mapped.filed_date).dt.days
assert mapped.signal_date.is_unique and len(mapped)==len(signals)==1008 and mapped.known.all() and mapped.source_age_days.ge(1).all()
assert events.effective_signal_date.min()<signals.min()
frame=mapped.set_index('signal_date')[['top5','HHI','known','source_age_days']]
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5','HHI']],'meta':frame[['known','source_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'late_train':tuple(plan['late_train_span'])})
prepared=dataset.prepare('late_train',col_set=['feature','meta']);assert len(prepared)==len(signals) and int(prepared[('meta','known')].sum())==1008
audit={'status':'sixteen_original_report_events_complete_2019_2022_late_train_Qlib_source_map','source_plan_sha256':sha(card/'plan.json'),'report_events':len(events),'late_train_Qlib_sessions':len(signals),'known_signal_sessions':int(mapped.known.sum()),'unknown_prefix_sessions':int((~mapped.known).sum()),'first_source_effective_signal':str(events.effective_signal_date.min().date()),'first_late_train_source_accession':str(mapped.accession.iloc[0]),'last_effective_signal':str(events.effective_signal_date.max().date()),'max_source_age_days':int(mapped.source_age_days.max()),'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'full_Stage1_training_coverage_qualified':False,'late_2019_2022_concentration_source_coverage_qualified':True}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('late_train',col_set=['feature','meta']))==len(signals)
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_2018_2022_concentration_source_bridge',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={'2018_Sep':s['2018_Sep_numeric_result_sha256'],'prior_bridge':s['prior_bridge_summary_sha256']})
 rec.save_objects(artifact_path='analysis',source_events=events,mapped_signals=mapped,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=len(events),late_train_sessions=len(signals),known_source_sessions=int(mapped.known.sum()),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 assert len(checked.load_object('metadata/dataset').prepare('late_train',col_set=['feature','meta']))==len(signals)
 assert len(checked.load_object('analysis/source_events'))==16
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_2018_2022_concentration_bridge','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-2018-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'2019_2022_original_reported_concentration_late_train_source_coverage_complete_earlier_Stage1_pending','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','report_events':16,'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:audit[k] for k in ('report_events','late_train_Qlib_sessions','known_signal_sessions','unknown_prefix_sessions','first_source_effective_signal','first_late_train_source_accession','max_source_age_days')});print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'report_events': 16, 'late_train_Qlib_sessions': 1008, 'known_signal_sessions': 1008, 'unknown_prefix_sessions': 0, 'first_source_effective_signal': '2018-12-10', 'first_late_train_source_accession': '0001193125-18-344229', 'max_source_age_days': 171}
{'recorder_id': 'b9f1a84df25846dc8bc45b2e95906b31', 'nas_sha256': 'b73b8d47847fa71b8340a9759dfa4d958dfb3858ed9cd56f7c3e87f19201189b'}


## 中文结论

16 个原始 SOXX 披露事件按 SEC 申报日之后的真实 Qlib 交易日生效；新增 2018-09 报告于 2018-12-10 生效，使 2019–2022 原 Stage1 训练 1,008 日的普通股集中度来源均已知，原先 41 日未知前缀消失。最大申报日龄 171 天。DatasetH/Recorder `b9f1a84df25846dc8bc45b2e95906b31` 与 NAS SHA-256 `b73b8d47847fa71b8340a9759dfa4d958dfb3858ed9cd56f7c3e87f19201189b` 定点回读通过。0 收益读取、0 拟合、0 账户；2011–2018 来源、成分股价格及 Stage1 0bp SOXX 账户仍待完成。